In [10]:
# ── RECOVERY CELL — run this if kernel restarts ───────────────────────────────
import numpy as np
import pickle
import pandas as pd

# Load processed splits
X_train_processed = np.load('../model/artifacts/X_train.npy', allow_pickle=True)
X_test_processed  = np.load('../model/artifacts/X_test.npy',  allow_pickle=True)
y_train           = np.load('../model/artifacts/y_train.npy', allow_pickle=True)
y_test            = np.load('../model/artifacts/y_test.npy',  allow_pickle=True)

# Load preprocessor and metadata
with open('../model/artifacts/preprocessor.pkl', 'rb') as f:
    preprocessor = pickle.load(f)

with open('../model/artifacts/scale_pos_weight.pkl', 'rb') as f:
    scale_pos_weight = pickle.load(f)

with open('../model/artifacts/feature_columns.pkl', 'rb') as f:
    col_order = pickle.load(f)

print(f"X_train : {X_train_processed.shape}")
print(f"X_test  : {X_test_processed.shape}")
print(f"y_train : {y_train.shape}")
print(f"scale_pos_weight : {scale_pos_weight:.4f}")


X_train : (5634, 23)
X_test  : (1409, 23)
y_train : (5634,)
scale_pos_weight : 2.7686


In [17]:
# Cell 1: Load raw data + preprocessing (FIXED)

import numpy as np
import pandas as pd
import pickle

# Load dataset
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# ── Basic cleaning ───────────────────────────────────────────────────────────
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)
df.drop(columns=['customerID', 'TotalCharges'], inplace=True)

# ── Binary encoding ──────────────────────────────────────────────────────────
binary_map = {
    'Yes': 1, 'No': 0,
    'Male': 1, 'Female': 0,
    'No phone service': 0,
    'No internet service': 0
}
df.replace(binary_map, inplace=True)

# ── Contract encoding ────────────────────────────────────────────────────────
contract_map = {
    'Month-to-month': 0,
    'One year': 1,
    'Two year': 2
}
df['Contract'] = df['Contract'].map(contract_map)

# ── One-hot encoding ─────────────────────────────────────────────────────────
df = pd.get_dummies(df, columns=['InternetService', 'PaymentMethod'])

# ❌ DO NOT encode Churn again (already done via replace)

# ── Final checks ─────────────────────────────────────────────────────────────
print(f"df shape : {df.shape}")
print("Churn unique values:", df['Churn'].unique())  # should be [0, 1]
print("NaN in Churn:", df['Churn'].isna().sum())     # should be 0

print("✅ Cell 1 fixed and ready")

df shape : (7032, 24)
Churn unique values: [0 1]
NaN in Churn: 0
✅ Cell 1 fixed and ready


In [18]:
# Cell 2: Engineer 8 new features

# ── 1. tenure_monthly_ratio ───────────────────────────────────────────────────
# High ratio = long tenure, low bill = loyal low-risk customer
df['tenure_monthly_ratio'] = df['tenure'] / (df['MonthlyCharges'] + 1)

# ── 2. is_long_term_contract ──────────────────────────────────────────────────
# Two-year contract customers almost never churn
df['is_long_term_contract'] = (df['Contract'] == 2).astype(int)

# ── 3. service_count ──────────────────────────────────────────────────────────
# Total number of add-on services subscribed
service_cols = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df['service_count'] = df[service_cols].sum(axis=1)

# ── 4. avg_monthly_per_service ────────────────────────────────────────────────
# High bill but few services = potential churn trigger
df['avg_monthly_per_service'] = df['MonthlyCharges'] / (df['service_count'] + 1)

# ── 5. is_senior_alone ────────────────────────────────────────────────────────
# Senior citizen with no partner and no dependents = high risk
df['is_senior_alone'] = (
    (df['SeniorCitizen'] == 1) &
    (df['Partner'] == 0) &
    (df['Dependents'] == 0)
).astype(int)

# ── 6. is_fiber_high_bill ─────────────────────────────────────────────────────
# Fiber optic + high bill = dissatisfied customer pattern
median_bill = df['MonthlyCharges'].median()
df['is_fiber_high_bill'] = (
    (df['InternetService_Fiber optic'] == 1) &
    (df['MonthlyCharges'] > median_bill)
).astype(int)

# ── 7. no_support_services ────────────────────────────────────────────────────
# No security + no tech support = more likely to leave when issues arise
df['no_support_services'] = (
    (df['OnlineSecurity'] == 0) &
    (df['TechSupport'] == 0)
).astype(int)

# ── 8. contract_tenure_interact ───────────────────────────────────────────────
# Interaction: long contract + long tenure = very stable customer
df['contract_tenure_interact'] = df['Contract'] * df['tenure']

# ── Verify ────────────────────────────────────────────────────────────────────
new_features = [
    'tenure_monthly_ratio', 'is_long_term_contract', 'service_count',
    'avg_monthly_per_service', 'is_senior_alone', 'is_fiber_high_bill',
    'no_support_services', 'contract_tenure_interact'
]

print("New features added:")
print(df[new_features].describe().round(3))
print(f"\ndf shape after engineering: {df.shape}  (should be 7032, 32)")

New features added:
       tenure_monthly_ratio  is_long_term_contract  is_senior_alone  \
count              7032.000               7032.000         7032.000   
mean                  0.631                  0.240            0.080   
std                   0.697                  0.427            0.271   
min                   0.010                  0.000            0.000   
25%                   0.148                  0.000            0.000   
50%                   0.458                  0.000            0.000   
75%                   0.769                  0.000            0.000   
max                   3.547                  1.000            1.000   

       is_fiber_high_bill  no_support_services  contract_tenure_interact  
count            7032.000             7032.000                  7032.000  
mean                0.417                0.579                    36.158  
std                 0.493                0.494                    50.618  
min                 0.000               

In [19]:
# Cell 3 fix — drop NaN in Churn before split
print(f"NaN in Churn before drop: {df['Churn'].isna().sum()}")

df = df.dropna(subset=['Churn'])

print(f"df shape after dropna: {df.shape}")

# Now re-split
X = df.drop(columns=['Churn'])
y = df['Churn']

print(f"NaN in y: {y.isna().sum()}  (should be 0)")

NaN in Churn before drop: 0
df shape after dropna: (7032, 32)
NaN in y: 0  (should be 0)


In [20]:
# Cell 3: Save feature_engineering.py + updated train/test splits

import os
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler

# ── 0. SAFETY CHECKS ─────────────────────────────────────────────────────────
print("df shape before split:", df.shape)
print("NaN in Churn:", df['Churn'].isna().sum())

# ── 1. Save feature_engineering.py ───────────────────────────────────────────
feature_engineering_code = '''
import pandas as pd

def engineer_features(df):
    service_cols = [
        'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
        'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
    ]

    df = df.copy()

    df['tenure_monthly_ratio']     = df['tenure'] / (df['MonthlyCharges'] + 1)
    df['is_long_term_contract']    = (df['Contract'] == 2).astype(int)
    df['service_count']            = df[service_cols].sum(axis=1)
    df['avg_monthly_per_service']  = df['MonthlyCharges'] / (df['service_count'] + 1)
    df['is_senior_alone']          = (
        (df['SeniorCitizen'] == 1) &
        (df['Partner'] == 0) &
        (df['Dependents'] == 0)
    ).astype(int)

    median_bill = df['MonthlyCharges'].median()
    df['is_fiber_high_bill']       = (
        (df['InternetService_Fiber optic'] == 1) &
        (df['MonthlyCharges'] > median_bill)
    ).astype(int)

    df['no_support_services']      = (
        (df['OnlineSecurity'] == 0) &
        (df['TechSupport'] == 0)
    ).astype(int)

    df['contract_tenure_interact'] = df['Contract'] * df['tenure']

    return df
'''

os.makedirs('../model', exist_ok=True)
with open('../model/feature_engineering.py', 'w') as f:
    f.write(feature_engineering_code)

print("✅ feature_engineering.py saved")

# ── 2. Train-Test Split ──────────────────────────────────────────────────────
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── 3. Preprocessing ─────────────────────────────────────────────────────────
continuous_cols_fe = [
    'tenure', 'MonthlyCharges',
    'tenure_monthly_ratio', 'avg_monthly_per_service',
    'service_count', 'contract_tenure_interact'
]

binary_cols_fe = [c for c in X_train_fe.columns if c not in continuous_cols_fe]

preprocessor_fe = ColumnTransformer(
    transformers=[
        ('scale', RobustScaler(), continuous_cols_fe),
        ('pass',  'passthrough',  binary_cols_fe)
    ],
    remainder='drop'
)

X_train_fe_processed = preprocessor_fe.fit_transform(X_train_fe)
X_test_fe_processed  = preprocessor_fe.transform(X_test_fe)

print(f"X_train_fe shape : {X_train_fe_processed.shape}")
print(f"X_test_fe shape  : {X_test_fe_processed.shape}")

# ── 4. Save Artifacts ────────────────────────────────────────────────────────
os.makedirs('../model/artifacts', exist_ok=True)

col_order_fe = continuous_cols_fe + binary_cols_fe

np.save('../model/artifacts/X_train_fe.npy', X_train_fe_processed)
np.save('../model/artifacts/X_test_fe.npy',  X_test_fe_processed)
np.save('../model/artifacts/y_train_fe.npy', y_train_fe.values)
np.save('../model/artifacts/y_test_fe.npy',  y_test_fe.values)

with open('../model/artifacts/preprocessor_fe.pkl', 'wb') as f:
    pickle.dump(preprocessor_fe, f)

with open('../model/artifacts/feature_columns_fe.pkl', 'wb') as f:
    pickle.dump(col_order_fe, f)

print("✅ All updated artifacts saved")
print(f"\nFinal feature count : {len(col_order_fe)} (23 original + 8 engineered = 31)")

df shape before split: (7032, 32)
NaN in Churn: 0
✅ feature_engineering.py saved
X_train_fe shape : (5625, 31)
X_test_fe shape  : (1407, 31)
✅ All updated artifacts saved

Final feature count : 31 (23 original + 8 engineered = 31)
